# Schottky Barrier Height (SBH)

Calculate the Schottky barrier heights of a labeled metal/semiconductor interface using a DFT workflow on the Mat3ra platform.

The barriers come from the potential lineup method. The Fermi level is taken from the interface calculation, where metal and semiconductor share it. The valence band maximum of the semiconductor is aligned to the interface through the macroscopically averaged electrostatic potential (ESP) at the same semiconductor plane in the interface and in the isolated semiconductor:

$$\Phi_p = \left[E_F - \bar{V}_{\mathrm{interface}}\right] - \left[E_{\mathrm{VBM}} - \bar{V}_{\mathrm{semiconductor}}\right], \qquad \Phi_n = E_{\mathrm{gap}} - \Phi_p$$

<h2 style="color:green">Usage</h2>

1. Create and save a metal/semiconductor interface with labels (for example via `create_interface_with_min_strain_zsl.ipynb`).
1. Set the interface and calculation parameters in cells 1.2 and 1.3 below, including which part of the interface is the semiconductor.
1. Click "Run" > "Run All" to run all cells.
1. Wait for the job to complete.
1. Scroll down to view the Schottky barrier heights.

## Summary

1. Set up the environment and parameters: install packages (JupyterLite only) and configure parameters for the interface, workflow, compute resources, and job.
1. Authenticate and initialize API client: authenticate via browser, initialize the client, then select account and project.
1. Create materials: load an interface from the `../uploads` folder, extract its semiconductor part using interface labels, strip labels required by Quantum ESPRESSO, and save both materials to the platform tagged with the interface name.
1. Configure workflow: select application, load the Schottky barrier workflow from Standata, set model and computational parameters, select the semiconductor plane used for the lineup, and preview the workflow.
1. Configure compute: get list of clusters and create compute configuration with selected cluster, queue, and number of processors.
1. Create the job with materials and workflow configuration: assemble the job from materials, workflow, project, and compute configuration.
1. Submit the job and monitor the status: submit the job and wait for completion.
1. Retrieve results: get and display the Schottky barrier heights and the average ESP profiles.

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples")

### 1.2. Set parameters
Provide the INTERFACE_NAME to match the name of the structure in "uploads" folder for search. The semiconductor part is extracted based on the substrate/film labels present in the generated interface; the other part is the metal.

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName
from mat3ra.made.tools.convert.interface_parts_enum import InterfacePartsEnum

# 2. Auth and organization parameters
ORGANIZATION_NAME = None

# 3. Material parameters
FOLDER = "../uploads"
INTERFACE_NAME = "Interface"  # To search for in "uploads" folder, account's materials, then Standata
RELAXED_INTERFACE_NAME = None  # To search for in "uploads" folder, account's materials, then Standata. Defaults to the interface
SEMICONDUCTOR_PART = InterfacePartsEnum.FILM
INTERFACE_SYSTEM_NAME = None  # Used as tag to group the materials. Defaults to shorthand from the loaded interface name

# 4. Workflow parameters
APPLICATION_NAME = "espresso"
WORKFLOW_SEARCH_TERM = "schottky_barrier.json"
MY_WORKFLOW_NAME = "SBH"

# 5. Compute parameters
CLUSTER_NAME = None
QUEUE_NAME = QueueName.D
PPN = 1

# 6. Job parameters
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 60  # seconds

### 1.3. Set specific SBH parameters
The interface is metallic, so its SCF uses smearing occupations; the smearing type and width can be set in `ADDITIONAL_PARAMETERS["system"]`.

An interface slab is asymmetric, so without a dipole correction a spurious field spans the cell and shifts the potential lineup; the region from `emaxpos` to `emaxpos + eopreg` must lie inside the vacuum.

Semi-local DFT underestimates band gaps, and with them the n-type barrier. Set `BAND_GAP` to a measured or hybrid-functional gap to use it instead of the DFT gap of the semiconductor; it moves only the conduction band, so the p-type barrier stays the DFT value.

In [ ]:
# Method parameters
PSEUDOPOTENTIAL_TYPE = "us"  # "us" (ultrasoft), "nc" (norm-conserving), "paw"
FUNCTIONAL = "pbe"  # for gga: "pbe", "pbesol"; for lda: "pz"
MODEL_SUBTYPE = "gga"

# K-grid and k-path
SCF_KGRID = None  # e.g. [8, 8, 1]
KPATH = None  # e.g. [{"point": "Γ", "steps": 20}, {"point": "M", "steps": 20}]

# Set SCF diagonalization, mixing and occupations in QE input
ADDITIONAL_PARAMETERS = {
    "electrons": {
        "diagonalization": "david",  # "david" or "cg"
        "mixing_beta": 0.3
    },
    # "system": {"smearing": "mv", "degauss": 0.02},
}
# Dipole correction along z; emaxpos to emaxpos + eopreg (crystal units) inside the vacuum:
# ADDITIONAL_PARAMETERS["control"] = {"tefield": True, "dipfield": True}
# ADDITIONAL_PARAMETERS["system"] = {**ADDITIONAL_PARAMETERS.get("system", {}), "edir": 3, "eamp": 0.0, "eopreg": 0.05, "emaxpos": 0.95}

# Energy cutoffs
ECUTWFC = 40
ECUTRHO = 200

# Band gap used for the n-type barrier, in eV. None uses the DFT gap of the semiconductor.
BAND_GAP = None

## 2. Authenticate and initialize API client
### 2.1. Authenticate
Authenticate in the browser and have credentials stored in environment variable "OIDC_ACCESS_TOKEN".

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Create materials
### 3.1. Load interface from local folder

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import load_material
from mat3ra.notebooks_utils.core.entity.material.analysis import get_interface_part_by_z_split, translate_slab_to_z_center
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize

interface = translate_slab_to_z_center(load_material(client, FOLDER, INTERFACE_NAME, ACCOUNT_ID))
relaxed_interface = translate_slab_to_z_center(load_material(client, FOLDER, RELAXED_INTERFACE_NAME, ACCOUNT_ID)) if RELAXED_INTERFACE_NAME else interface

visualize(relaxed_interface, repetitions=[1, 1, 1])
visualize(relaxed_interface, repetitions=[1, 1, 1], rotation="-90x")

### 3.2. Create materials from interface parts
The semiconductor slab is isolated based on labels, then labels removed as they are not compatible with Quantum ESPRESSO. The two materials (interface, semiconductor) are named and visualized.

In [ ]:
import re

interface_shorthand_match = re.match(r"^(.+?)\s", interface.name)
interface_system_name = INTERFACE_SYSTEM_NAME or (
    interface_shorthand_match.group(1) if interface_shorthand_match else INTERFACE_NAME
)
semiconductor_material = get_interface_part_by_z_split(interface, relaxed_interface, SEMICONDUCTOR_PART)
interface_material = relaxed_interface.clone()

semiconductor_material.basis.set_labels_from_list([])
interface_material.basis.set_labels_from_list([])

interface_material.name = f"{interface_system_name} Interface"
semiconductor_material.name = f"{interface_system_name} Semiconductor"

materials_by_role = {"interface": interface_material, "semiconductor": semiconductor_material}
for role, material in materials_by_role.items():
    print(f"  {role}: {material.name}")
visualize(list(materials_by_role.values()), repetitions=[1, 1, 1], rotation="-90x")

### 3.3. Save materials

In [ ]:
from mat3ra.made.material import Material

saved_materials = {}
for role, material in materials_by_role.items():
    material_config = material.to_dict()
    material_config["name"] = material.name
    existing_tags = material_config.get("tags") or []
    material_config["tags"] = sorted(set([*existing_tags, interface_system_name]))
    saved = Material.create(client.materials.create(material_config, owner_id=ACCOUNT_ID))
    saved_materials[role] = saved
    print(f"  {role}: {saved.name} ({saved.id}) | tags={material_config['tags']}")

## 4. Configure workflow
### 4.1. Select application

In [ ]:
from mat3ra.ade.application import Application
from mat3ra.standata.applications import ApplicationStandata

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)
print(f"Using application: {app.name}")

### 4.2. Load workflow from Standata and preview it

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow

workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(WORKFLOW_SEARCH_TERM)
workflow = Workflow.create(workflow_config)

workflow.name = MY_WORKFLOW_NAME

visualize_workflow(workflow)

### 4.3. Set model and its parameters (physics)

In [ ]:
from mat3ra.mode import ModelFactory
from mat3ra.standata.model_tree import ModelTreeStandata

model_config = ModelTreeStandata.get_model_by_parameters(
    type="dft", subtype=MODEL_SUBTYPE, functional=FUNCTIONAL
)
model_config["method"] = {"type": "pseudopotential", "subtype": PSEUDOPOTENTIAL_TYPE}
model = ModelFactory.create(model_config)

for subworkflow in workflow.subworkflows:
    if subworkflow.application.name == APPLICATION_NAME:
        subworkflow.model = model

### 4.4. Modify method (computational parameters): k-grid, k-path, cutoffs, diagonalization, mixing, and occupations
The interface and the semiconductor share the in-plane lattice, so one SCF k-grid applies to both.

In [ ]:
from mat3ra.wode.context.providers import PointsPathDataProvider
from mat3ra.notebooks_utils.workflow import apply_planewave_cutoffs, apply_scf_kgrid, patch_workflow_qe_input

scf_unit_names = ["pw_scf", "pw_bands"]

apply_scf_kgrid(workflow, SCF_KGRID, material=interface_material)

if KPATH is not None:
    for subworkflow in workflow.subworkflows:
        if unit := subworkflow.get_unit_by_name(name="pw_bands"):
            unit.add_context(PointsPathDataProvider(path=KPATH, isEdited=True).get_context_item_data())
            subworkflow.set_unit(unit)

if ECUTWFC is not None:
    for unit_name in scf_unit_names:
        apply_planewave_cutoffs(workflow, ECUTWFC, ECUTRHO, unit_name=unit_name)

if ADDITIONAL_PARAMETERS:
    patch_workflow_qe_input(workflow, ADDITIONAL_PARAMETERS, unit_names=scf_unit_names)

### 4.5. Select the semiconductor plane for the potential lineup and the band gap
The workflow lines up the same semiconductor plane in the interface and in the isolated semiconductor: the outermost ESP minimum along z when the semiconductor is the film, the lowest detected one when it is the substrate.

In [ ]:
from mat3ra.notebooks_utils.workflow import set_assignment_value

semiconductor_esp_minimum_index = -1 if SEMICONDUCTOR_PART == InterfacePartsEnum.FILM else 0
set_assignment_value(workflow, "Set Semiconductor ESP Minimum Index", str(semiconductor_esp_minimum_index))

if BAND_GAP is not None:
    set_assignment_value(workflow, "Set Semiconductor Band Gap", str(BAND_GAP))


### 4.6. Preview final workflow

In [ ]:
visualize_workflow(workflow)

## 5. Create the compute configuration
### 5.1. Get list of clusters

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")

### 5.2. Create compute configuration

In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
else:
    cluster = clusters[0]

compute = Compute(
    cluster=cluster,
    queue=QUEUE_NAME,
    ppn=PPN
)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}")

## 6. Create the job
### 6.1. Create job

In [ ]:
from mat3ra.notebooks_utils.job import create_job
from mat3ra.utils.namespace import dict_to_namespace_recursive
from mat3ra.notebooks_utils.ui import display_JSON

materials = list(saved_materials.values())

job_name = f"{MY_WORKFLOW_NAME} {interface_system_name} {timestamp}"
workflow.name = job_name

print(f"Materials: {[m.id for m in materials]}")
print(f"Project: {project_id}")

job_response = create_job(
    api_client=client,
    materials=materials,
    workflow=workflow,
    project_id=project_id,
    owner_id=ACCOUNT_ID,
    prefix=job_name,
    compute=compute.to_dict(),
)

job = dict_to_namespace_recursive(job_response)
job_id = job._id
print("✅ Job created successfully!")
print(f"Job ID: {job_id}")

display_JSON(job_response)

## 7. Submit the job and monitor the status

In [ ]:
client.jobs.submit(job_id)
print(f"✅ Job {job_id} submitted successfully!")

In [ ]:
from mat3ra.notebooks_utils.api.job import wait_for_jobs_to_finish_async

await wait_for_jobs_to_finish_async(client.jobs, [job_id], poll_interval=POLL_INTERVAL)

## 8. Retrieve and visualize results
### 8.1. Schottky barrier heights

In [ ]:
from mat3ra.notebooks_utils.core.entity.property.job import get_fermi_energy_flowchart_id

job_data = client.jobs.get(job_id)
job_scope = {key: value for item in job_data["scopeTrack"] for key, value in item["scope"]["global"].items()}

fermi_energy_unit_id = get_fermi_energy_flowchart_id(job_data)
fermi_energy = client.properties.get_for_job(job_id, property_name="fermi_energy", unit_id=fermi_energy_unit_id)[0]
print(f"Fermi energy (interface): {fermi_energy['value']:.3f} eV")
print(f"Valence band maximum (semiconductor): {job_scope['VBM']:.3f} eV")
print(f"Band gap: {job_scope['BAND_GAP']:.3f} eV")
print(f"p-type Schottky barrier height: {job_scope['SCHOTTKY_BARRIER_P']:.3f} eV")
print(f"n-type Schottky barrier height: {job_scope['SCHOTTKY_BARRIER_N']:.3f} eV")

### 8.2. Average ESP profiles

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.property.visualize import visualize_properties

avg_esp_results = {}
for subworkflow in job_data["workflow"]["subworkflows"]:
    for unit in subworkflow["units"]:
        if "average_potential_profile" in [result["name"] for result in unit.get("results", [])]:
            avg_esp_results[subworkflow["name"]] = client.properties.get_for_job(
                job_id,
                property_name="average_potential_profile",
                unit_id=unit["flowchartId"],
            )[0]

for subworkflow_name, avg_esp_data in avg_esp_results.items():
    visualize_properties(avg_esp_data, title=subworkflow_name)

### 8.3. Plot the interface ESP profile with the metal and semiconductor regions and the lineup

In [ ]:
from mat3ra.notebooks_utils.plot import configure_matplotlib_renderer

configure_matplotlib_renderer()
import matplotlib.pyplot as plt

metal_part = InterfacePartsEnum.SUBSTRATE if SEMICONDUCTOR_PART == InterfacePartsEnum.FILM else InterfacePartsEnum.FILM
regions = {"Metal": get_interface_part_by_z_split(interface, relaxed_interface, metal_part), "Semiconductor": semiconductor_material.clone()}

interface_profile = avg_esp_results["Avg ESP (Interface)"]
plt.figure(figsize=(10, 6))
plt.plot(interface_profile["xDataArray"], interface_profile["yDataSeries"][1], label="Macroscopic Average Potential", linewidth=2)
for (region_name, region_material), color in zip(regions.items(), ["gold", "blue"]):
    region_material.to_cartesian()
    region_z = [coordinate[2] for coordinate in region_material.basis.coordinates.values]
    plt.axvspan(min(region_z), max(region_z), color=color, alpha=0.2, label=f"{region_name} Region")
lineup_esp = job_scope["AVG_ESP_INTERFACE"][job_scope["SEMICONDUCTOR_ESP_MINIMUM_INDEX"]]
plt.axhline(lineup_esp, color="blue", linestyle=":", linewidth=2, label=f"Semiconductor lineup ESP = {lineup_esp:.3f} eV")
plt.xlabel("z-coordinate (A)", fontsize=12)
plt.ylabel("Macroscopic Average Potential (eV)", fontsize=12)
plt.title(f"Schottky barrier heights: p-type {job_scope['SCHOTTKY_BARRIER_P']:.3f} eV, n-type {job_scope['SCHOTTKY_BARRIER_N']:.3f} eV", fontsize=14, fontweight="bold")
plt.legend(loc="best", fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()